In [3]:
import hashlib
import os
import re
import warnings
from dataclasses import dataclass, field
from typing import Optional, Sequence
import cvxpy as cp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyvinecopulib as pv
import yfinance as yf
from arch import arch_model
from joblib import Parallel, delayed
from scipy.optimize import minimize
from scipy.special import logsumexp
from scipy.stats import genpareto, qmc, rankdata

In [4]:
@dataclass
class ModelParams:
  scale_factor: float = 100.0
  lags: int = 3
  evt_q: float = 0.90
  xi_max: Optional[float] = None
  n_steps: int = 21
  sobol_pow: int = 18
  seed: Optional[int] = 0
  n_jobs: int = 1
  vine_threads: int = 1
  vine_families: Optional[Sequence] = None
  vine_trunc: Optional[int] = None
  cache_dir: Optional[str] = None
  cvar_q: float = 0.95
  gamma_var: float = 1.0
  gamma_cvar: float = 1.0
  lambda_anchor: float = 0.0
  min_weight: float = 0.0
  max_weight: float = 0.40
  cov_shrink: float = 0.0
  opt_scale: float = 100.0
  view_tol: float = 1e-5
  strict_views: bool = True
  min_ens_ratio: float = 0.02

In [5]:
@dataclass
class FitState:
  fit: bool = False
  best_models: dict = field(default_factory=dict)
  best_fits: dict = field(default_factory=dict)
  best_lags: dict = field(default_factory=dict)
  ar_history: dict = field(default_factory=dict)
  vol_init: dict = field(default_factory=dict)
  r_init: dict = field(default_factory=dict)
  marginal_distributions: dict = field(default_factory=dict)
  vine_structure: object = None
  copula: object = None

In [6]:
@dataclass
class TailFit:
  c_L: float
  scale_L: float
  c_U: float
  scale_U: float
  p_l: float
  p_u: float
  u_lower: float
  u_upper: float
  ecdf_data: np.ndarray

In [7]:
@dataclass
class EPResult:
  q: np.ndarray
  mu: pd.Series
  cov: pd.DataFrame
  kl: float
  ens: float

In [8]:
def horizon_returns(R, horizon: Optional[int] = None) -> np.ndarray:
  R = np.asarray(R, dtype=float)
  if R.ndim == 2:
    if horizon not in (None, 1):
      raise ValueError("2D returns are taken as-is; `horizon` only applies to 3D input.")
    return R
  if R.ndim != 3:
    raise ValueError(f"Expected 2D or 3D returns, got {R.ndim}D.")
  T = R.shape[2]
  h = T if horizon is None else int(horizon)
  if not 1 <= h <= T:
    raise ValueError(f"horizon={h} outside simulated range 1..{T}.")
  return np.prod(1.0 + R[:, :, :h], axis=2) - 1.0

In [9]:
def tail_average(L, q, prob=None) -> np.ndarray:
  L = np.asarray(L, dtype=float)
  if L.ndim == 1:
    L = L[:, None]
  S = L.shape[0]
  prob = np.full(S, 1.0 / S) if prob is None else np.asarray(prob, float).ravel()
  prob = prob / prob.sum()
  alpha = 1.0 - q
  order = np.argsort(-L, axis=0)
  Ls = np.take_along_axis(L, order, axis=0)
  ps = prob[order]
  prev = np.cumsum(ps, axis=0) - ps
  take = np.clip(alpha - prev, 0.0, ps)
  return (take * Ls).sum(axis=0) / alpha

In [10]:
class DataStore:
  def __init__(self, debug: bool = False):
    self.debug = debug

  def download(self, tickers, start, end, interval):
    data = {}
    for ticker in tickers:
      t = ticker.strip()
      try:
        s = yf.Ticker(t).history(start=start, end=end, interval=interval)["Close"]
        if s.empty:
          print(f"Warning: No data returned for {t}")
          continue
        s.index = pd.to_datetime(s.index).tz_localize(None).normalize()
        data[t] = s[~s.index.duplicated(keep="last")]
      except Exception as e:
        print(f"Failed to download {t}: {e}")
    if not data:
      raise ValueError("No data was successfully downloaded for any of the provided tickers.")
    return pd.concat(data, axis=1).sort_index()

  def get_data(self, tickers, start, end, interval="1d", benchmark="^GSPC", ffill_limit=3):
    self.benchmark_ticker = benchmark
    tag = hashlib.md5(("|".join(sorted(map(str, tickers))) + interval).encode()).hexdigest()[:8]
    df_path = f"portfolio_{tag}_{start}_{end}.parquet"
    bm_path = f"benchmark_{benchmark.replace('^', '')}_{start}_{end}.parquet"

    if not (os.path.exists(df_path) and os.path.exists(bm_path)):
      df = self.download(tickers, start, end, interval)
      bench = self.download([benchmark], start, end, interval)
      df.to_parquet(df_path); bench.to_parquet(bm_path)
    else:
      df, bench = pd.read_parquet(df_path), pd.read_parquet(bm_path)

    self.returns_raw = df.ffill(limit=ffill_limit).pct_change().dropna()
    self.benchmark = bench.pct_change().dropna()
    self.universe = self.returns_raw.columns
    return self.returns_raw, self.benchmark

  def plot_data(self):
    (np.cumsum(self.returns_raw * 100, axis=0) + 100).plot(figsize=(15, 10)); plt.show()

  def plot_benchmark(self):
    (np.cumsum(self.benchmark * 100, axis=0) + 100).plot(figsize=(15, 10)); plt.show()

In [11]:
class MarginalTailModel:
  def __init__(self, params: ModelParams, ticker):
    self.dist_p = params
    self.ticker = ticker
    self.tail_fit: Optional[TailFit] = None

  def _fit_gpd(self, exceed, side):
    if exceed.size < 10:
      raise ValueError(f"{self.ticker}: only {exceed.size} {side}-tail exceedances; "
                       "lower evt_q or use more history.")
    c, _, scale = genpareto.fit(exceed, floc=0)
    xi_max = self.dist_p.xi_max
    if xi_max is not None and c > xi_max:
      c, _, scale = genpareto.fit(exceed, f0=xi_max, floc=0)
    if c >= 1.0:
      print(f"{self.ticker}: {side}-tail GPD xi={c:.4f} >= 1 -> infinite mean, "
            "theoretical CVaR/ES is not finite.")
    elif c >= 0.5:
      print(f"{self.ticker}: {side}-tail GPD xi={c:.4f} is very heavy; "
            "CVaR can converge slowly even with QMC.")
    return c, scale

  def fit_dist(self, r: pd.Series) -> pd.Series:
    full_idx = r.index
    r = r.dropna()
    x = r.to_numpy(dtype=float)
    n = x.size

    q = self.dist_p.evt_q
    u_lower, u_upper = np.quantile(x, [1.0 - q, q])
    lower_mask = x < u_lower
    upper_mask = x > u_upper
    body_mask = ~(lower_mask | upper_mask)

    n_lo, n_up = int(lower_mask.sum()), int(upper_mask.sum())
    if n_lo == 0 or n_up == 0 or not body_mask.any():
      raise ValueError(f"{self.ticker}: insufficient observations for EVT split.")

    exc_lo = u_lower - x[lower_mask]
    exc_up = x[upper_mask] - u_upper
    c_L, scale_L = self._fit_gpd(exc_lo, "lower")
    c_U, scale_U = self._fit_gpd(exc_up, "upper")
    p_l, p_u = n_lo / n, n_up / n

    u = np.empty(n, dtype=float)
    u[lower_mask] = p_l * genpareto.sf(exc_lo, c_L, loc=0, scale=scale_L)
    body = x[body_mask]
    u[body_mask] = p_l + (1.0 - p_l - p_u) * rankdata(body, method="average") / (body.size + 1)
    u[upper_mask] = 1.0 - p_u * genpareto.sf(exc_up, c_U, loc=0, scale=scale_U)

    self.tail_fit = TailFit(c_L, scale_L, c_U, scale_U, p_l, p_u,
                            u_lower, u_upper, np.sort(body))
    return pd.Series(u, index=r.index).reindex(full_idx)

  def inverse_cdf(self, u_resid):
    t = self.tail_fit
    u = np.asarray(u_resid, dtype=float)
    out = np.empty_like(u)
    lo = u < t.p_l
    hi = u > 1.0 - t.p_u
    body = ~(lo | hi)

    x = t.ecdf_data
    nb = x.size
    p_grid = t.p_l + (1.0 - t.p_l - t.p_u) * np.arange(1, nb + 1) / (nb + 1)
    out[body] = np.interp(u[body], p_grid, x)
    out[lo] = t.u_lower - genpareto.isf(u[lo] / t.p_l, t.c_L, loc=0, scale=t.scale_L)
    out[hi] = t.u_upper + genpareto.isf((1.0 - u[hi]) / t.p_u, t.c_U, loc=0, scale=t.scale_U)
    return out

In [12]:
class GarchFitter:
  @staticmethod
  def fit_arch(model, warm_start_params):
    fit_kwargs = dict(disp="off", show_warning=False)
    if warm_start_params is not None:
      try:
        return model.fit(starting_values=np.asarray(warm_start_params, dtype=float),
                         **fit_kwargs)
      except (ValueError, TypeError, np.linalg.LinAlgError):
        pass
    return model.fit(**fit_kwargs)

  @staticmethod
  def select_model(r, lag_candidates, o, warm_params):
    best_bic, best = np.inf, (None, None, None)
    for lags in lag_candidates:
      model = arch_model(r, mean="AR", lags=lags, vol="GARCH",
                         p=1, o=o, q=1, dist="studentst")
      try:
        fit = GarchFitter.fit_arch(model, warm_params)
      except (ValueError, RuntimeError, np.linalg.LinAlgError):
        continue
      if fit.convergence_flag:
        continue
      pr = fit.params
      persistence = pr["alpha[1]"] + pr["beta[1]"] + 0.5 * pr.get("gamma[1]", 0.0)
      if persistence < 1 and fit.bic < best_bic:
        best_bic, best = fit.bic, (model, fit, lags)
    return best

  @staticmethod
  def fit_asset(r, max_lags, warm_params, warm_lags):
    if warm_lags is not None:
      res = GarchFitter.select_model(r, [warm_lags], 1, warm_params)
      if res[0] is not None:
        return res
    lags = range(1, max_lags + 1)
    res = GarchFitter.select_model(r, lags, 1, None)
    if res[0] is None:
      res = GarchFitter.select_model(r, lags, 0, None)
    if res[0] is None:
      raise ValueError("No stationary GARCH fit found")
    return res

In [15]:
class GarchEVTCopula(GarchFitter):
  _GARCH_KEYS = {"alpha[1]", "gamma[1]", "beta[1]"}

  def __init__(self, params: ModelParams, debug: bool = False):
    self.debug = debug
    self.model_p = params
    self.sobol_seed = params.seed
    self._sobol_cache = None
    self.model_fit: Optional[FitState] = None
    self.copula = None
    self.cols = None

  def _fit_ar_garch(self, returns: pd.DataFrame) -> pd.DataFrame:
    if self.debug:
      print("------------- Fitting AR-GARCH -------------")
    mf = self.model_fit
    cols = list(returns.columns)
    log_returns = np.log1p(returns)
    warm = bool(mf.fit)
    series = {c: (log_returns[c] * self.model_p.scale_factor).dropna() for c in cols}

    jobs = []
    for c in cols:
      wp = wl = None
      if warm and c in mf.best_fits:
        wp, wl = mf.best_fits[c].params, mf.best_lags[c]
      jobs.append(delayed(GarchFitter.fit_asset)(series[c], self.model_p.lags, wp, wl))
    results = Parallel(n_jobs=self.model_p.n_jobs)(jobs)

    resid = {}
    for c, (model, fit, lags) in zip(cols, results):
      hist = series[c].to_numpy(dtype=float)
      if hist.size < lags:
        raise ValueError(f"{c}: need at least {lags} observations for the AR recursion.")

      mf.best_models[c] = model
      mf.best_fits[c] = fit
      mf.best_lags[c] = lags
      mf.ar_history[c] = hist[-lags:].copy()
      mf.vol_init[c] = float(np.asarray(fit.conditional_volatility)[-1])
      mf.r_init[c] = hist[-1]
      resid[c] = pd.Series(np.asarray(fit.std_resid), index=series[c].index)
    return pd.DataFrame(resid, index=returns.index, columns=cols, dtype=float)

  def _get_uniform_residuals(self, residuals: pd.DataFrame) -> pd.DataFrame:
    if self.debug:
      print("------------ Uniform Residuals -------------")
    u = {}
    for ticker in residuals.columns:
      m = MarginalTailModel(self.model_p, ticker)
      u[ticker] = m.fit_dist(residuals[ticker])
      self.model_fit.marginal_distributions[ticker] = m
    return pd.DataFrame(u, index=residuals.index, columns=residuals.columns)

  def _fit_vine_copula(self, u_resid: pd.DataFrame, structure=None):
    if self.debug:
      print("----------- Fitting Vine Copula ------------")
    clean = u_resid.dropna(how="any")
    if clean.empty:
      raise ValueError("No complete residual rows available for the vine fit.")
    eps = 1e-10
    u = np.asfortranarray(np.clip(clean.to_numpy(dtype=np.float64), eps, 1 - eps))

    kw = dict(num_threads=self.model_p.vine_threads)
    if self.model_p.vine_families is not None:
      kw["family_set"] = self.model_p.vine_families
    if self.model_p.vine_trunc is not None:
      kw["trunc_lvl"] = self.model_p.vine_trunc
    controls = pv.FitControlsVinecop(**kw)
    self.copula = pv.Vinecop.from_data(u, controls, structure=structure)

  def fit(self, returns: pd.DataFrame, state: FitState) -> FitState:
    self.model_fit = state
    z_resid = self._fit_ar_garch(returns)
    self.cols = z_resid.columns
    u_resid = self._get_uniform_residuals(z_resid)

    structure = state.vine_structure if state.fit else None
    self._fit_vine_copula(u_resid, structure=structure)

    state.vine_structure = self.copula.structure
    state.copula = self.copula
    state.fit = True
    return state

  def _base_uniforms(self, m, n_steps, n_assets):
    key = (m, n_steps, n_assets, self.sobol_seed)
    if self._sobol_cache is not None and self._sobol_cache[0] == key:
      return self._sobol_cache[1]
    engine = qmc.Sobol(d=n_steps * n_assets, scramble=True, seed=self.sobol_seed)
    u = np.clip(engine.random_base2(m=m), 1e-12, 1 - 1e-12)
    self._sobol_cache = (key, u)
    return u

  def _simulate(self, n_assets):
    n_steps, m = self.model_p.n_steps, self.model_p.sobol_pow
    n = 2 ** m
    base = self._base_uniforms(m, n_steps, n_assets).reshape(n * n_steps, n_assets)
    u = self.copula.inverse_rosenblatt(np.asfortranarray(base),
                                       num_threads=self.model_p.vine_threads)
    return np.asarray(u).reshape(n, n_steps, n_assets)

  def _inverse_semi_parametric_cdf(self, uniform_samples):
    if self.debug:
      print("-------- Inverse Semi Parametric CDF -------")
    n_paths, n_steps, n_assets = uniform_samples.shape
    z = np.empty((n_assets, n_steps, n_paths), dtype=float)
    eps = 1e-6
    for i, ticker in enumerate(self.cols):
      model = self.model_fit.marginal_distributions[ticker]
      u_i = np.clip(uniform_samples[:, :, i].T, eps, 1 - eps)
      z[i] = model.inverse_cdf(u_i)
    return z

  @staticmethod
  def _ar_coefficients(params, p):
    pat = re.compile(r"^.+\[(\d+)\]$")
    out = {}
    for key in params.index:
      if key in GarchEVTCopula._GARCH_KEYS:
        continue
      m = pat.match(key)
      if m:
        out[int(m.group(1))] = float(params[key])
    if sorted(out) != list(range(1, p + 1)):
      raise KeyError(f"Expected AR lags 1..{p}, found {sorted(out)} in {list(params.index)}")
    return np.array([out[k] for k in range(1, p + 1)])

  def _convert_to_returns(self, z):
    if self.debug:
      print("------------ Converting to Returns ---------")
    mf = self.model_fit
    A, T, N = z.shape
    lags = [int(mf.best_lags[c]) for c in self.cols]
    P = max(lags)

    const = np.zeros((A, 1)); omega = np.zeros((A, 1))
    alpha = np.zeros((A, 1)); gamma = np.zeros((A, 1)); beta = np.zeros((A, 1))
    phi = np.zeros((A, P))
    sigma0 = np.zeros(A); eps0 = np.zeros(A)
    buf = np.zeros((A, P + T, N))

    for i, (c, p) in enumerate(zip(self.cols, lags)):
      prm = mf.best_fits[c].params
      const[i] = float(prm.get("Const", 0.0))
      omega[i] = float(prm["omega"])
      alpha[i] = float(prm["alpha[1]"])
      gamma[i] = float(prm.get("gamma[1]", 0.0))
      beta[i] = float(prm["beta[1]"])
      phi[i, P - p:] = self._ar_coefficients(prm, p)[::-1]
      hist = np.asarray(mf.ar_history[c], dtype=float)
      if hist.size != p:
        raise ValueError(f"{c}: AR history size {hist.size} != selected p={p}")
      buf[i, P - p:P, :] = hist[:, None]
      sigma0[i] = mf.vol_init[c]
      eps0[i] = float(np.asarray(mf.best_fits[c].resid)[-1])

    sigma2 = np.repeat((sigma0 ** 2)[:, None], N, axis=1)
    eps_prev = np.repeat(eps0[:, None], N, axis=1)
    for t in range(T):
      sigma2 = omega + (alpha + gamma * (eps_prev < 0.0)) * eps_prev ** 2 + beta * sigma2
      np.maximum(sigma2, 1e-14, out=sigma2)
      eps_t = np.sqrt(sigma2) * z[:, t, :]
      ar = np.einsum("ap,apn->an", phi, buf[:, t:t + P, :])
      buf[:, P + t, :] = const + ar + eps_t
      eps_prev = eps_t
    return buf[:, P:, :]

  def generate_sample(self) -> np.ndarray:
    if self.debug:
      print("------------ Generating Samples ------------")
    u = self._simulate(len(self.cols))
    z = self._inverse_semi_parametric_cdf(u)
    log_r = self._convert_to_returns(z) / self.model_p.scale_factor
    sim_r = np.ascontiguousarray(np.expm1(log_r).transpose(2, 0, 1))
    if not np.all(np.isfinite(sim_r)):
      raise FloatingPointError("Simulation produced NaN/inf returns. "
                               "Inspect EVT xi and GARCH path diagnostics.")
    return sim_r

In [16]:
class CVaR(GarchEVTCopula):
  def __init__(self, params: ModelParams, debug: bool = False):
    super().__init__(params, debug)
    self.sim_path = None
    self.start = self.end = None

  def _make_sim_path(self) -> str:
    p = self.model_p
    key = repr((tuple(map(str, self.cols)), p.scale_factor, p.lags, p.evt_q, p.xi_max,
                None if p.vine_families is None else tuple(map(str, p.vine_families)),
                p.vine_trunc))
    digest = hashlib.md5(key.encode()).hexdigest()[:10]
    seed = "random" if p.seed is None else str(p.seed)
    name = (f"simulation_{pd.Timestamp(self.start):%Y%m%d}_{pd.Timestamp(self.end):%Y%m%d}"
            f"_steps{p.n_steps}_sobol2p{p.sobol_pow}_seed{seed}_{digest}.npy")
    return os.path.join(p.cache_dir, name) if p.cache_dir else name

  def fit_garch_evt_copula(self, returns: pd.DataFrame, state: Optional[FitState] = None,
                           refit: bool = True) -> FitState:
    state = state if state is not None else FitState()
    self.start, self.end = returns.index[0], returns.index[-1]
    self.universe = returns.columns

    if refit or not (state.fit and state.copula is not None):
      state = self.fit(returns, state)
    else:
      self.model_fit, self.copula, self.cols = state, state.copula, returns.columns
    self.sim_path = self._make_sim_path()
    return state

  def generate_r_sample(self, force_resimulate: bool = False) -> np.ndarray:
    if self.sim_path is None:
      raise ValueError("Fit the model first!")
    use_cache = (self.sobol_seed is not None and not force_resimulate
                 and os.path.exists(self.sim_path))
    if use_cache:
      return np.load(self.sim_path)
    sim = self.generate_sample()
    if self.sobol_seed is not None:
      np.save(self.sim_path, sim)
    return sim

  def to_horizon_returns(self, R, horizon: Optional[int] = None):
    return horizon_returns(R, horizon)

  def _asset_names(self, R, n_assets):
    if isinstance(R, pd.DataFrame):
      return list(R.columns)
    if self.cols is not None and len(self.cols) == n_assets:
      return list(self.cols)
    return list(range(n_assets))

  def cvar(self, R, w=None, q=None, horizon=None, prob=None) -> pd.Series:
    q = self.model_p.cvar_q if q is None else q
    H = horizon_returns(R, horizon)
    out = pd.Series(tail_average(-H, q, prob), index=self._asset_names(R, H.shape[1]),
                    name="CVaR")
    if w is not None:
      out["portfolio"] = tail_average(-H @ np.asarray(w, float).ravel(), q, prob)[0]
    return out

  def mcvar(self, R, w, q=None, horizon=None, prob=None) -> pd.Series:
    q = self.model_p.cvar_q if q is None else q
    H = horizon_returns(R, horizon)
    w = np.asarray(w, float).ravel()
    S = H.shape[0]
    prob = np.full(S, 1.0 / S) if prob is None else np.asarray(prob, float) / np.sum(prob)
    alpha = 1.0 - q
    order = np.argsort(-(-H @ w))
    ps = prob[order]
    prev = np.cumsum(ps) - ps
    take = np.zeros(S)
    take[order] = np.clip(alpha - prev, 0.0, ps)
    return pd.Series(take @ (-H) / alpha, index=self._asset_names(R, H.shape[1]),
                     name="MCVaR")

  def verify_cvar(self, R, w, b=None, horizon=None, prob=None, verbose=True) -> dict:
    w = np.asarray(w, float).ravel()
    mcvar = self.mcvar(R, w, horizon=horizon, prob=prob)
    cvar = self.cvar(R, w, horizon=horizon, prob=prob)["portfolio"]
    rc = w * mcvar.to_numpy()
    out = dict(mcvar=mcvar.to_numpy(), cvar=cvar, rc=rc, euler_sum=rc.sum(),
               realized_budget=rc / cvar)
    if b is not None:
      b = np.asarray(b, float).ravel(); b = b / b.sum()
      out["max_budget_error"] = float(np.max(np.abs(out["realized_budget"] - b)))
    if verbose:
      for k, v in out.items():
        print(f"{k:>17}: {v}")
    return out

In [17]:
class LambdaSolver:
  def posterior(self, prior, A, b, k_eq, k_ineq):
    prior = np.asarray(prior, float)
    prior = prior / prior.sum()
    logp = np.log(prior)

    m = A @ prior
    s = np.sqrt(((A - m[:, None]) ** 2) @ prior)
    s = np.where(s > 1e-14, s, 1.0)
    As, bs = (A - m[:, None]) / s[:, None], (b - m) / s
    K = As.shape[0]

    def dual(lam):
      z = logp - lam @ As
      lse = logsumexp(z)
      return lse + lam @ bs, bs - As @ np.exp(z - lse)

    bounds = [(None, None)] * k_eq + [(0.0, None)] * k_ineq
    res = minimize(dual, np.zeros(K), jac=True, method="L-BFGS-B", bounds=bounds,
                   options=dict(maxiter=5000, maxfun=20000, ftol=1e-16, gtol=1e-11))
    lam = res.x
    q = self._q_from_lambda(logp, As, lam)
    if self._violation(q, As, bs, k_eq) > 1e-6:
      lam = self._solve_cvxpy(logp, As, bs, k_eq, k_ineq)
      q = self._q_from_lambda(logp, As, lam)
    self.last_lambda = lam
    return q

  @staticmethod
  def _q_from_lambda(logp, As, lam):
    z = logp - lam @ As
    return np.exp(z - logsumexp(z))

  @staticmethod
  def _violation(q, As, bs, k_eq):
    r = As @ q - bs
    eq = np.abs(r[:k_eq]).max() if k_eq else 0.0
    ineq = max(r[k_eq:].max(), 0.0) if r.size > k_eq else 0.0
    return max(eq, ineq)

  @staticmethod
  def _solve_cvxpy(logp, As, bs, k_eq, k_ineq):
    lam = cp.Variable(As.shape[0])
    cons = [lam[k_eq:] >= 0] if k_ineq > 0 else []
    prob = cp.Problem(cp.Minimize(cp.log_sum_exp(logp - lam @ As) + lam @ bs), cons)
    prob.solve(solver=cp.CLARABEL)
    if prob.status not in ("optimal", "optimal_inaccurate"):
      raise RuntimeError(f"Entropy-pooling dual failed: {prob.status}. "
                         "The views are probably infeasible for this scenario set.")
    return lam.value

In [18]:
class ViewHandlers:
  @staticmethod
  def _compute_theta(R, q):
    mu = q @ R
    return {"mu": mu, "var": q @ (R - mu) ** 2}

  def _scale_views(self, views, horizon_days, period_days, compound=True):
    f = horizon_days / period_days
    out = []
    for v in views:
      v = dict(v)
      if v["view"] == "mean":
        v["value"] = (1 + v["value"]) ** f - 1 if compound else v["value"] * f
      elif v["view"] == "relative":
        v["value"] = v["value"] * f
      elif v["view"] == "volatility":
        v["value"] = v["value"] * np.sqrt(f)
      out.append(v)
    return out

  def _view_mean(self, view, R, columns):
    return (R[:, columns.get_loc(view["ticker"])], view["value"],
            view["type"], view.get("direction", "le"))

  def _view_relative(self, view, R, columns):
    i, j = columns.get_loc(view["ticker1"]), columns.get_loc(view["ticker2"])
    return R[:, i] - R[:, j], view["value"], view["type"], view.get("direction", "le")

  def _view_volatility(self, view, R, columns, theta):
    i = columns.get_loc(view["ticker"])
    return ((R[:, i] - theta["mu"][i]) ** 2, view["value"] ** 2,
            view["type"], view.get("direction", "le"))

  def _view_correlation(self, view, R, columns, theta):
    i, j = columns.get_loc(view["ticker1"]), columns.get_loc(view["ticker2"])
    cov_target = view["value"] * np.sqrt(theta["var"][i] * theta["var"][j])
    A = (R[:, i] - theta["mu"][i]) * (R[:, j] - theta["mu"][j])
    return A, cov_target, view["type"], view.get("direction", "le")

  def _sort_views(self, views):
    sorted_views = {"mean": [], "relative": [], "volatility": [], "correlation": []}
    for view in views:
      if view["view"] not in sorted_views:
        raise ValueError(f"Unsupported view type: {view['view']!r}")
      sorted_views[view["view"]].append(view)
    return sorted_views

In [19]:
class EntropyPooling(ViewHandlers, LambdaSolver):
  def __init__(self, params: ModelParams, debug: bool = False, heuristic: str = "H1"):
    if heuristic not in ("H1", "H2"):
      raise ValueError("heuristic must be 'H1' or 'H2'")
    self.p, self.debug, self.heuristic = params, debug, heuristic

  def _get_view_matrices(self, views, R, theta, columns):
    A_eq, b_eq, A_ineq, b_ineq = [], [], [], []
    fix_mean, fix_vol = {}, {}
    handlers = {
      "mean": lambda v: self._view_mean(v, R, columns),
      "relative": lambda v: self._view_relative(v, R, columns),
      "volatility": lambda v: self._view_volatility(v, R, columns, theta),
      "correlation": lambda v: self._view_correlation(v, R, columns, theta),
    }
    eq_mean = {v["ticker"] for v in views if v["view"] == "mean" and v["type"] == "eq"}
    eq_vol = {v["ticker"] for v in views if v["view"] == "volatility" and v["type"] == "eq"}

    for view in views:
      A, b, vtype, direction = handlers[view["view"]](view)
      if vtype == "eq":
        A_eq.append(A); b_eq.append(b)
      elif direction == "ge":
        A_ineq.append(-A); b_ineq.append(-b)
      else:
        A_ineq.append(A); b_ineq.append(b)

      if view["view"] == "volatility":
        fix_mean[view["ticker"]] = None
      elif view["view"] == "correlation":
        for t in (view["ticker1"], view["ticker2"]):
          fix_mean[t] = None
          fix_vol[t] = None

    for t in fix_mean:
      if t not in eq_mean:
        i = columns.get_loc(t)
        A_eq.append(R[:, i]); b_eq.append(theta["mu"][i])
    for t in fix_vol:
      if t not in eq_vol:
        i = columns.get_loc(t)
        A_eq.append((R[:, i] - theta["mu"][i]) ** 2); b_eq.append(theta["var"][i])
    return A_eq, b_eq, A_ineq, b_ineq

  @staticmethod
  def _stack(A_eq, b_eq, A_ineq, b_ineq):
    A_list, b_list = [], []
    for A, b in ((A_eq, b_eq), (A_ineq, b_ineq)):
      if len(A) > 0:
        A_list.append(np.atleast_2d(A)); b_list.append(np.atleast_1d(b))
    k_eq, k_ineq = len(A_eq), len(A_ineq)
    if not A_list:
      return None, None, 0, 0
    return np.vstack(A_list), np.concatenate(b_list), k_eq, k_ineq

  def _check_views(self, q, R, columns, views):
    th = self._compute_theta(R, q)
    bad = []
    for v in views:
      kind = v["view"]
      if kind == "mean":
        val = th["mu"][columns.get_loc(v["ticker"])]
      elif kind == "volatility":
        val = np.sqrt(th["var"][columns.get_loc(v["ticker"])])
      elif kind == "relative":
        val = th["mu"][columns.get_loc(v["ticker1"])] - th["mu"][columns.get_loc(v["ticker2"])]
      else:
        i, j = columns.get_loc(v["ticker1"]), columns.get_loc(v["ticker2"])
        val = (q * (R[:, i] - th["mu"][i]) * (R[:, j] - th["mu"][j])).sum() \
              / np.sqrt(th["var"][i] * th["var"][j])
      tol = max(self.p.view_tol, 1e-4 * abs(v["value"]))
      if v["type"] == "eq":
        ok = abs(val - v["value"]) <= tol
      elif v.get("direction", "le") == "ge":
        ok = val >= v["value"] - tol
      else:
        ok = val <= v["value"] + tol
      if not ok:
        bad.append((v, float(val)))
    if bad:
      msg = "Views not satisfied by posterior: " + "; ".join(
        f"{v} -> posterior {val:.6g}" for v, val in bad)
      if self.p.strict_views:
        raise ValueError(msg)
      warnings.warn(msg)

  @staticmethod
  def _moments(R, q, columns):
    mu = q @ R
    dev = R - mu
    cov = (dev.T * q) @ dev
    return pd.Series(mu, index=columns), pd.DataFrame(cov, index=columns, columns=columns)

  def solve(self, R, columns, views=None, p=None, horizon: Optional[int] = None,
            view_period_days: Optional[int] = None) -> EPResult:
    columns = pd.Index(columns)
    H = horizon_returns(R, horizon)
    if not np.isfinite(H).all():
      raise ValueError("returns contain NaN/inf; clean before entropy pooling")
    S, N = H.shape
    h = (np.asarray(R).shape[2] if horizon is None else horizon) if np.ndim(R) == 3 else 1

    if p is None:
      p = np.full(S, 1.0 / S)
    else:
      p = np.asarray(p, float).ravel()
      if len(p) != S:
        raise ValueError(f"Expected {S} prior probabilities, got {len(p)}")
      if (p < 0).any():
        raise ValueError("prior probabilities must be non-negative")
      p = p / p.sum()
    p = np.clip(p, 1e-300, None)

    views_d = []
    if views:
      views_d = (self._scale_views(views, h, view_period_days) if view_period_days
                 else [dict(v) for v in views])

    q = p
    if views_d:
      sv = self._sort_views(views_d)
      stages = [sv["mean"] + sv["relative"], sv["volatility"], sv["correlation"]]
      applied = []
      theta = self._compute_theta(H, q)
      for stage in stages:
        if not stage:
          continue
        applied += stage
        prior = p if self.heuristic == "H1" else q
        A_eq, b_eq, A_in, b_in = self._get_view_matrices(applied, H, theta, columns)
        A, b, k_eq, k_ineq = self._stack(A_eq, b_eq, A_in, b_in)
        q = self.posterior(prior, A, b, k_eq, k_ineq)
        theta = self._compute_theta(H, q)
      self._check_views(q, H, columns, views_d)

    q = np.asarray(q, float).ravel()
    pos = q > 0
    kl = float(np.sum(q[pos] * np.log(q[pos] / p[pos])))
    ens = float(np.exp(-np.sum(q[pos] * np.log(q[pos]))))
    if ens / S < self.p.min_ens_ratio:
      warnings.warn(f"Posterior effective sample size {ens:.0f} of {S} "
                    f"({ens / S:.1%}): views are very aggressive for this scenario set.")
    if self.debug:
      print(f"[EP] {self.heuristic}  KL(q||p)={kl:.4f}  ENS={ens:.0f}/{S}")

    mu, cov = self._moments(H, q, columns)
    return EPResult(q=q, mu=mu, cov=cov, kl=kl, ens=ens)

In [20]:
class Optimizer:
  def __init__(self, params: ModelParams, debug: bool = False):
    self.p, self.debug = params, debug

  def _cov_factor(self, cov):
    C = np.asarray(cov, float)
    C = 0.5 * (C + C.T)
    if self.p.cov_shrink > 0:
      C = (1 - self.p.cov_shrink) * C + self.p.cov_shrink * np.diag(np.diag(C))
    ev, V = np.linalg.eigh(C)
    ev = np.clip(ev, max(ev.max(), 0.0) * 1e-10 + 1e-16, None)
    return V * np.sqrt(ev)

  def optimize_w(self, returns, mean, covariance, w_prev=None, scenario_prob=None):
    P = self.p
    R = np.asarray(returns, float)
    S, N = R.shape
    if P.max_weight * N < 1.0 - 1e-12 or P.min_weight * N > 1.0 + 1e-12:
      raise ValueError("Weight bounds are infeasible for sum(w)=1.")

    c = P.opt_scale
    q = np.full(S, 1.0 / S) if scenario_prob is None else np.asarray(scenario_prob, float).ravel()
    q = q / q.sum()

    Rs = R * c
    mu_s = np.asarray(mean, float).ravel() * c
    L_s = self._cov_factor(covariance) * c

    w, u, zeta = cp.Variable(N), cp.Variable(S), cp.Variable()
    es = zeta + (q @ u) / (1.0 - P.cvar_q)
    constraints = [u >= -Rs @ w - zeta, u >= 0, cp.sum(w) == 1,
                   w >= P.min_weight, w <= P.max_weight]

    obj = mu_s @ w - (P.gamma_var / c) * cp.sum_squares(L_s.T @ w) - P.gamma_cvar * es
    if w_prev is not None and P.lambda_anchor > 0:
      obj = obj - c * P.lambda_anchor * cp.sum_squares(w - np.asarray(w_prev, float))

    prob = cp.Problem(cp.Maximize(obj), constraints)
    last = None
    for solver in (cp.CLARABEL, cp.SCS):
      try:
        prob.solve(solver=solver)
      except cp.SolverError as e:
        last = str(e); continue
      last = prob.status
      if prob.status in ("optimal", "optimal_inaccurate"):
        break
    else:
      raise ValueError(f"MVO+CVaR optimisation failed: {last}")

    out = np.clip(w.value, 0.0, None)
    return out / out.sum()


In [21]:
class Portfolio(DataStore):
  def __init__(self, params: ModelParams, debug: bool = False, heuristic: str = "H1"):
    super().__init__(debug=debug)
    self.p = params
    self.risk = CVaR(params, debug)
    self.ep = EntropyPooling(params, debug, heuristic)
    self.opt = Optimizer(params, debug)
    self.fit_state = FitState()
    self.R_sim = None
    self.last_ep: Optional[EPResult] = None

  def fit(self, returns: pd.DataFrame, refit: bool = True) -> FitState:
    self.fit_state = self.risk.fit_garch_evt_copula(returns, self.fit_state, refit=refit)
    self.R_sim = None
    return self.fit_state

  def simulate(self, force: bool = False) -> np.ndarray:
    self.R_sim = self.risk.generate_r_sample(force_resimulate=force)
    return self.R_sim

  @staticmethod
  def apply_lot_sizing(weights, asset_prices, total_portfolio_value: float = 1000.0):
    w = np.asarray(weights, float).copy()
    prices = np.asarray(asset_prices, float)
    valid = np.isfinite(prices) & (prices > 0)
    w[~valid] = 0.0
    if w.sum() <= 0:
      raise ValueError("No investable asset with a valid price.")
    w /= w.sum()
    px = np.where(valid, prices, np.inf)
    ideal = w * total_portfolio_value / px
    shares = np.floor(ideal)
    cash = total_portfolio_value - float(np.sum(np.where(valid, shares * prices, 0.0)))
    while True:
      gap = ideal - shares
      ok = valid & (px <= cash + 1e-12) & (gap > 0.5)
      if not ok.any():
        break
      j = int(np.argmax(np.where(ok, gap * px, -np.inf)))
      shares[j] += 1.0
      cash -= px[j]
    spent = np.where(valid, shares * prices, 0.0)
    return (spent / spent.sum() if spent.sum() > 0 else w), shares

  def optimize_portfolio(self, views, returns=None, horizon: Optional[int] = None,
                         p=None, w_prev=None, view_period_days: Optional[int] = None,
                         round_weights: bool = False, prices=None,
                         portfolio_value: float = 1000.0):
    if returns is None:
      if self.R_sim is None:
        self.simulate()
      R, columns = self.R_sim, pd.Index(self.risk.cols)
    else:
      R = returns.values if isinstance(returns, pd.DataFrame) else np.asarray(returns, float)
      columns = (returns.columns if isinstance(returns, pd.DataFrame)
                 else pd.Index(self.risk.cols))

    H = horizon_returns(R, horizon)
    self.last_horizon_returns = H
    ep = self.ep.solve(R, columns, views=views, p=p, horizon=horizon,
                       view_period_days=view_period_days)
    self.last_ep = ep
    if self.debug:
      print(f"posterior mean:\n{ep.mu}\nposterior covariance:\n{ep.cov}")

    w_raw = self.opt.optimize_w(H, ep.mu, ep.cov, w_prev=w_prev, scenario_prob=ep.q)
    self.last_cvar = self.risk.cvar(H, w_raw, prob=ep.q)

    if not round_weights:
      return w_raw, columns
    if prices is None:
      raise ValueError("round_weights=True needs `prices` (latest asset prices).")
    if isinstance(prices, pd.Series):
      prices = prices.reindex(columns).values
    w_opt, self.last_shares = self.apply_lot_sizing(w_raw, prices, portfolio_value)
    return w_opt, columns